In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

from src.config import (
    SAMPLE_SUBMISSION_PATH,
    SUBMISSION_DIR,
    TARGET,
    ID_COL,
    RANDOM_STATE,
    N_SPLITS
)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\pc\Desktop\yzta-2026-datathon


In [2]:
def rmse(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return np.sqrt(mse)

In [3]:
PREDICTION_ROOT = PROJECT_ROOT / "predictions"

oof_files = sorted(PREDICTION_ROOT.glob("*/*_oof.csv"))
test_files = sorted(PREDICTION_ROOT.glob("*/*_test.csv"))

print("OOF file count:", len(oof_files))
print("Test file count:", len(test_files))

for path in oof_files:
    print("OOF:", path.relative_to(PROJECT_ROOT))

for path in test_files:
    print("TEST:", path.relative_to(PROJECT_ROOT))

OOF file count: 27
Test file count: 27
OOF: predictions\catboost\catboost_base_oof.csv
OOF: predictions\catboost\catboost_depth4_lr003_oof.csv
OOF: predictions\catboost\catboost_depth5_lr003_oof.csv
OOF: predictions\catboost\catboost_depth6_l2_5_oof.csv
OOF: predictions\catboost\catboost_depth6_lr002_oof.csv
OOF: predictions\catboost\catboost_depth7_lr002_oof.csv
OOF: predictions\catboost_native\catboost_native_depth4_l2_5_oof.csv
OOF: predictions\catboost_native\catboost_native_depth4_lr002_oof.csv
OOF: predictions\catboost_native\catboost_native_depth4_oof.csv
OOF: predictions\catboost_native\catboost_native_depth5_oof.csv
OOF: predictions\extra_regressors\adaboost_tree_oof.csv
OOF: predictions\extra_regressors\bagging_tree_oof.csv
OOF: predictions\extra_regressors\elasticnet_oof.csv
OOF: predictions\extra_regressors\extra_trees_tuned_oof.csv
OOF: predictions\extra_regressors\gradient_boosting_tuned_oof.csv
OOF: predictions\extra_regressors\hist_gradient_boosting_oof.csv
OOF: predict

In [4]:
oof_predictions = {}
test_predictions = {}

y_true = None
train_ids = None
test_ids = None

for oof_path in oof_files:
    group_name = oof_path.parent.name
    model_name = oof_path.name.replace("_oof.csv", "")
    full_model_name = f"{group_name}__{model_name}"

    df_oof = pd.read_csv(oof_path)

    if y_true is None:
        y_true = df_oof["y_true"].values
        train_ids = df_oof[ID_COL].values
    else:
        assert np.allclose(y_true, df_oof["y_true"].values), f"y_true mismatch: {full_model_name}"
        assert np.all(train_ids == df_oof[ID_COL].values), f"train id mismatch: {full_model_name}"

    oof_predictions[full_model_name] = df_oof["oof_pred"].values


for test_path in test_files:
    group_name = test_path.parent.name
    model_name = test_path.name.replace("_test.csv", "")
    full_model_name = f"{group_name}__{model_name}"

    df_test = pd.read_csv(test_path)

    if test_ids is None:
        test_ids = df_test[ID_COL].values
    else:
        assert np.all(test_ids == df_test[ID_COL].values), f"test id mismatch: {full_model_name}"

    test_predictions[full_model_name] = df_test["test_pred"].values


print("Loaded OOF models:", len(oof_predictions))
print("Loaded TEST models:", len(test_predictions))

assert set(oof_predictions.keys()) == set(test_predictions.keys()), "OOF ve TEST model listeleri aynı değil!"

Loaded OOF models: 27
Loaded TEST models: 27


In [5]:
common_models = sorted(set(oof_predictions.keys()) & set(test_predictions.keys()))

print("Common model count:", len(common_models))
common_models

Common model count: 27


['catboost__catboost_base',
 'catboost__catboost_depth4_lr003',
 'catboost__catboost_depth5_lr003',
 'catboost__catboost_depth6_l2_5',
 'catboost__catboost_depth6_lr002',
 'catboost__catboost_depth7_lr002',
 'catboost_native__catboost_native_depth4',
 'catboost_native__catboost_native_depth4_l2_5',
 'catboost_native__catboost_native_depth4_lr002',
 'catboost_native__catboost_native_depth5',
 'extra_regressors__adaboost_tree',
 'extra_regressors__bagging_tree',
 'extra_regressors__elasticnet',
 'extra_regressors__extra_trees_tuned',
 'extra_regressors__gradient_boosting_tuned',
 'extra_regressors__hist_gradient_boosting',
 'extra_regressors__random_forest_tuned',
 'extra_regressors__ridge',
 'lgbm_xgb__lightgbm_base',
 'lgbm_xgb__lightgbm_leaves15',
 'lgbm_xgb__lightgbm_leaves63',
 'lgbm_xgb__lightgbm_lr002',
 'lgbm_xgb__lightgbm_regularized',
 'lgbm_xgb__xgboost_base',
 'lgbm_xgb__xgboost_depth3',
 'lgbm_xgb__xgboost_depth4_lr002',
 'lgbm_xgb__xgboost_regularized']

In [6]:
model_scores = []

for model_name in common_models:
    pred = np.clip(oof_predictions[model_name], 0, 10)
    score = rmse(y_true, pred)

    model_scores.append({
        "model": model_name,
        "rmse": score
    })

model_scores_df = pd.DataFrame(model_scores)
model_scores_df = model_scores_df.sort_values("rmse").reset_index(drop=True)

model_scores_df

,model,rmse
0,catboost_native__catboost_native_depth4,1.216596
1,catboost_native__catboost_native_depth4_lr002,1.216782
2,catboost_native__catboost_native_depth4_l2_5,1.216888
3,catboost_native__catboost_native_depth5,1.217939
4,catboost__catboost_depth4_lr003,1.219307
5,catboost__catboost_depth6_lr002,1.221311
6,catboost__catboost_depth5_lr003,1.221557
7,catboost__catboost_depth6_l2_5,1.222720
8,catboost__catboost_base,1.222916
9,catboost__catboost_depth7_lr002,1.223179


In [7]:
best_single_model = model_scores_df.loc[0, "model"]
best_single_rmse = model_scores_df.loc[0, "rmse"]

best_single_test_pred = np.clip(test_predictions[best_single_model], 0, 10)

submission_best_single = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: best_single_test_pred
})

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

best_single_path = SUBMISSION_DIR / f"submission_best_single_{best_single_model}.csv"

submission_best_single.to_csv(best_single_path, index=False)

print("Best single model:", best_single_model)
print("Best single RMSE:", best_single_rmse)
print("Saved:", best_single_path)

display(submission_best_single.head())

Best single model: catboost_native__catboost_native_depth4
Best single RMSE: 1.2165962843966693
Saved: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_best_single_catboost_native__catboost_native_depth4.csv


,id,bilissel_performans_skoru
0,1,5.989866
1,2,6.434160
2,3,3.037792
3,4,7.112258
4,5,3.621443


In [8]:
simple_blend_results = []

max_n = min(12, len(common_models))

for n in range(2, max_n + 1):
    selected_models = model_scores_df.head(n)["model"].tolist()

    blend_oof = np.mean(
        [oof_predictions[m] for m in selected_models],
        axis=0
    )

    blend_oof = np.clip(blend_oof, 0, 10)
    score = rmse(y_true, blend_oof)

    simple_blend_results.append({
        "n_models": n,
        "models": selected_models,
        "rmse": score
    })

simple_blend_results_df = pd.DataFrame(simple_blend_results)
simple_blend_results_df = simple_blend_results_df.sort_values("rmse").reset_index(drop=True)

simple_blend_results_df

,n_models,models,rmse
0,5,"[catboost_native__catboost_native_depth4, catb...",1.216051
1,6,"[catboost_native__catboost_native_depth4, catb...",1.216166
2,3,"[catboost_native__catboost_native_depth4, catb...",1.216338
3,2,"[catboost_native__catboost_native_depth4, catb...",1.216361
4,4,"[catboost_native__catboost_native_depth4, catb...",1.216362
5,7,"[catboost_native__catboost_native_depth4, catb...",1.216459
6,8,"[catboost_native__catboost_native_depth4, catb...",1.216764
7,9,"[catboost_native__catboost_native_depth4, catb...",1.217067
8,12,"[catboost_native__catboost_native_depth4, catb...",1.217256
9,10,"[catboost_native__catboost_native_depth4, catb...",1.217339


In [9]:
best_simple_row = simple_blend_results_df.iloc[0]
best_simple_models = best_simple_row["models"]
best_simple_rmse = best_simple_row["rmse"]

best_simple_test_pred = np.mean(
    [test_predictions[m] for m in best_simple_models],
    axis=0
)

best_simple_test_pred = np.clip(best_simple_test_pred, 0, 10)

submission_simple_blend = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: best_simple_test_pred
})

simple_blend_path = SUBMISSION_DIR / "submission_best_simple_blend.csv"

submission_simple_blend.to_csv(simple_blend_path, index=False)

print("Best simple blend RMSE:", best_simple_rmse)
print("Best simple blend models:")
for m in best_simple_models:
    print("-", m)

print("Saved:", simple_blend_path)

display(submission_simple_blend.head())

Best simple blend RMSE: 1.2160514795641506
Best simple blend models:
- catboost_native__catboost_native_depth4
- catboost_native__catboost_native_depth4_lr002
- catboost_native__catboost_native_depth4_l2_5
- catboost_native__catboost_native_depth5
- catboost__catboost_depth4_lr003
Saved: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_best_simple_blend.csv


,id,bilissel_performans_skoru
0,1,5.979948
1,2,6.417566
2,3,3.011396
3,4,7.122070
4,5,3.643538


In [10]:
candidate_models = model_scores_df.head(6)["model"].tolist()

candidate_models

['catboost_native__catboost_native_depth4',
 'catboost_native__catboost_native_depth4_lr002',
 'catboost_native__catboost_native_depth4_l2_5',
 'catboost_native__catboost_native_depth5',
 'catboost__catboost_depth4_lr003',
 'catboost__catboost_depth6_lr002']

In [11]:
rng = np.random.default_rng(RANDOM_STATE)

weighted_results = []

n_trials = 5000

for i in range(n_trials):
    weights = rng.dirichlet(np.ones(len(candidate_models)))

    blend_oof = np.zeros(len(y_true))

    for w, model_name in zip(weights, candidate_models):
        blend_oof += w * oof_predictions[model_name]

    blend_oof = np.clip(blend_oof, 0, 10)
    score = rmse(y_true, blend_oof)

    result = {
        "rmse": score,
    }

    for model_name, weight in zip(candidate_models, weights):
        result[model_name] = weight

    weighted_results.append(result)

weighted_results_df = pd.DataFrame(weighted_results)
weighted_results_df = weighted_results_df.sort_values("rmse").reset_index(drop=True)

weighted_results_df.head(20)

,rmse,catboost_native__catboost_native_depth4,catboost_native__catboost_native_depth4_lr002,catboost_native__catboost_native_depth4_l2_5,catboost_native__catboost_native_depth5,catboost__catboost_depth4_lr003,catboost__catboost_depth6_lr002
0,1.215972,0.356506,0.244179,0.157257,0.007581,0.221439,0.013037
1,1.215978,0.360877,0.167455,0.163953,0.067138,0.214199,0.026378
2,1.215978,0.316543,0.306059,0.134079,0.017794,0.202037,0.023488
3,1.215979,0.394831,0.147621,0.192988,0.017631,0.208215,0.038713
4,1.215981,0.343367,0.284010,0.087377,0.003782,0.255748,0.025716
5,1.215986,0.289299,0.218760,0.207229,0.032177,0.184843,0.067693
6,1.215988,0.418324,0.239131,0.049085,0.080298,0.179522,0.033640
7,1.215989,0.489682,0.210010,0.039033,0.047003,0.183648,0.030624
8,1.215990,0.371275,0.125162,0.220036,0.061066,0.187549,0.034911
9,1.215991,0.320686,0.150710,0.173991,0.074207,0.242333,0.038072


In [12]:
best_weighted_row = weighted_results_df.iloc[0]
best_weighted_rmse = best_weighted_row["rmse"]

best_weighted_test_pred = np.zeros(len(test_ids))

print("Best weighted RMSE:", best_weighted_rmse)
print("Weights:")

for model_name in candidate_models:
    weight = best_weighted_row[model_name]
    print(model_name, ":", weight)

    best_weighted_test_pred += weight * test_predictions[model_name]

best_weighted_test_pred = np.clip(best_weighted_test_pred, 0, 10)

submission_weighted_blend = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: best_weighted_test_pred
})

weighted_blend_path = SUBMISSION_DIR / "submission_best_weighted_blend.csv"

submission_weighted_blend.to_csv(weighted_blend_path, index=False)

print("Saved:", weighted_blend_path)

display(submission_weighted_blend.head())

Best weighted RMSE: 1.2159719089246095
Weights:
catboost_native__catboost_native_depth4 : 0.3565061597411857
catboost_native__catboost_native_depth4_lr002 : 0.2441788403833292
catboost_native__catboost_native_depth4_l2_5 : 0.15725745167747388
catboost_native__catboost_native_depth5 : 0.007581430952012622
catboost__catboost_depth4_lr003 : 0.22143939745148308
catboost__catboost_depth6_lr002 : 0.01303671979451537
Saved: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_best_weighted_blend.csv


,id,bilissel_performans_skoru
0,1,5.984201
1,2,6.407862
2,3,3.012971
3,4,7.129546
4,5,3.639412


In [13]:
stack_models = model_scores_df.head(8)["model"].tolist()

stack_train = pd.DataFrame({
    model_name: oof_predictions[model_name]
    for model_name in stack_models
})

stack_test = pd.DataFrame({
    model_name: test_predictions[model_name]
    for model_name in stack_models
})

display(stack_train.head())
display(stack_test.head())

,catboost_native__catboost_native_depth4,catboost_native__catboost_native_depth4_lr002,catboost_native__catboost_native_depth4_l2_5,catboost_native__catboost_native_depth5,catboost__catboost_depth4_lr003,catboost__catboost_depth6_lr002,catboost__catboost_depth5_lr003,catboost__catboost_depth6_l2_5
0,0.286026,0.326254,0.386092,0.417754,0.317006,0.716165,0.387353,0.418661
1,6.585255,6.602222,6.573231,6.624978,6.492392,6.566120,6.635654,6.655761
2,4.951261,4.812762,5.022352,4.850613,5.272690,5.280265,5.289035,5.324308
3,8.333887,8.339909,8.243912,8.255469,8.259863,8.301649,8.414723,8.380686
4,2.928182,2.756442,2.640269,2.811548,2.435077,2.563292,2.556890,2.595478


,catboost_native__catboost_native_depth4,catboost_native__catboost_native_depth4_lr002,catboost_native__catboost_native_depth4_l2_5,catboost_native__catboost_native_depth5,catboost__catboost_depth4_lr003,catboost__catboost_depth6_lr002,catboost__catboost_depth5_lr003,catboost__catboost_depth6_l2_5
0,5.989866,5.953782,5.943266,5.975517,6.037308,5.995767,5.970050,5.956820
1,6.434160,6.427955,6.390240,6.481976,6.353497,6.405296,6.357755,6.445689
2,3.037792,3.064245,3.026880,3.009793,2.918269,2.816492,2.927075,2.881429
3,7.112258,7.113670,7.119924,7.088126,7.176370,7.244464,7.223409,7.214837
4,3.621443,3.650592,3.646934,3.647807,3.650912,3.630433,3.681610,3.748819


In [14]:
cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

stack_oof = np.zeros(len(y_true))
stack_test_folds = np.zeros((len(test_ids), N_SPLITS))
stack_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(stack_train, y_true), start=1):
    X_stack_train = stack_train.iloc[train_idx]
    X_stack_valid = stack_train.iloc[valid_idx]

    y_stack_train = y_true[train_idx]
    y_stack_valid = y_true[valid_idx]

    stack_model = Ridge(alpha=1.0)

    stack_model.fit(X_stack_train, y_stack_train)

    valid_pred = stack_model.predict(X_stack_valid)
    valid_pred = np.clip(valid_pred, 0, 10)

    stack_oof[valid_idx] = valid_pred

    fold_rmse = rmse(y_stack_valid, valid_pred)
    stack_fold_scores.append(fold_rmse)

    test_pred = stack_model.predict(stack_test)
    test_pred = np.clip(test_pred, 0, 10)

    stack_test_folds[:, fold - 1] = test_pred

    print(f"Fold {fold} RMSE:", fold_rmse)

stack_rmse_mean = np.mean(stack_fold_scores)
stack_rmse_std = np.std(stack_fold_scores)
stack_oof_rmse = rmse(y_true, stack_oof)

print("Stacking CV RMSE:", stack_rmse_mean, "±", stack_rmse_std)
print("Stacking OOF RMSE:", stack_oof_rmse)

Fold 1 RMSE: 1.2198171027860407
Fold 2 RMSE: 1.2171705737839067
Fold 3 RMSE: 1.2024647121295318
Fold 4 RMSE: 1.2097231771049408
Fold 5 RMSE: 1.2315290339086291
Stacking CV RMSE: 1.2161409199426099 ± 0.009793820929947412
Stacking OOF RMSE: 1.216180355081953


In [15]:
stack_test_pred = stack_test_folds.mean(axis=1)
stack_test_pred = np.clip(stack_test_pred, 0, 10)

submission_stack = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: stack_test_pred
})

stack_path = SUBMISSION_DIR / "submission_stacking_ridge.csv"

submission_stack.to_csv(stack_path, index=False)

print("Saved:", stack_path)

display(submission_stack.head())

Saved: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_stacking_ridge.csv


,id,bilissel_performans_skoru
0,1,5.999292
1,2,6.417568
2,3,3.006298
3,4,7.131297
4,5,3.633714


In [16]:
final_comparison = []

final_comparison.append({
    "method": f"best_single_{best_single_model}",
    "rmse": best_single_rmse,
    "file": str(best_single_path)
})

final_comparison.append({
    "method": "best_simple_blend",
    "rmse": best_simple_rmse,
    "file": str(simple_blend_path)
})

final_comparison.append({
    "method": "best_weighted_blend",
    "rmse": best_weighted_rmse,
    "file": str(weighted_blend_path)
})

final_comparison.append({
    "method": "stacking_ridge",
    "rmse": stack_oof_rmse,
    "file": str(stack_path)
})

final_comparison_df = pd.DataFrame(final_comparison)
final_comparison_df = final_comparison_df.sort_values("rmse").reset_index(drop=True)

final_comparison_df

,method,rmse,file
0,best_weighted_blend,1.215972,C:\Users\pc\Desktop\yzta-2026-datathon\submiss...
1,best_simple_blend,1.216051,C:\Users\pc\Desktop\yzta-2026-datathon\submiss...
2,stacking_ridge,1.216180,C:\Users\pc\Desktop\yzta-2026-datathon\submiss...
3,best_single_catboost_native__catboost_native_d...,1.216596,C:\Users\pc\Desktop\yzta-2026-datathon\submiss...


In [17]:
best_method = final_comparison_df.loc[0, "method"]

print("Best method:", best_method)

if best_method.startswith("best_single"):
    final_submission = submission_best_single.copy()

elif best_method == "best_simple_blend":
    final_submission = submission_simple_blend.copy()

elif best_method == "best_weighted_blend":
    final_submission = submission_weighted_blend.copy()

elif best_method == "stacking_ridge":
    final_submission = submission_stack.copy()

else:
    raise ValueError("Unknown best method")

final_best_path = SUBMISSION_DIR / "submission_ensemble_final_best.csv"

final_submission.to_csv(final_best_path, index=False)

print("Saved final best submission:", final_best_path)
display(final_submission.head())

Best method: best_weighted_blend
Saved final best submission: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_ensemble_final_best.csv


,id,bilissel_performans_skoru
0,1,5.984201
1,2,6.407862
2,3,3.012971
3,4,7.129546
4,5,3.639412


In [18]:
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Final submission shape:", final_submission.shape)
print("Sample submission shape:", sample_submission.shape)

print("\nFinal columns:")
print(final_submission.columns.tolist())

print("\nSample columns:")
print(sample_submission.columns.tolist())

display(final_submission.head())
display(sample_submission.head())

Final submission shape: (24000, 2)
Sample submission shape: (2, 2)

Final columns:
['id', 'bilissel_performans_skoru']

Sample columns:
['id', 'bilissel_performans_skoru']


,id,bilissel_performans_skoru
0,1,5.984201
1,2,6.407862
2,3,3.012971
3,4,7.129546
4,5,3.639412


,id,bilissel_performans_skoru
0,1,7.85
1,2,4.32
